In [ ]:
#Download Dataset
import os

# Set API
os.environ['KAGGLE_API_TOKEN'] = "KGAT_40c900374d46e1e88f4b3c1c3d17d2f7"

!pip install -q librosa soundfile numpy kaggle

# DDowwnload data and unzip dataset
print("Downloading Cats vs Dogs Audio dataset...")
!kaggle datasets download -d stealthtechnologies/cats-vs-dogs-audio-classification
print("Unzipping dataset...")
!unzip -q cats-vs-dogs-audio-classification.zip -d /content/dataset
print("Dataset is ready at: /content/dataset")

# Mount Google Drive to save model checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Model Architecture
import numpy as np
import librosa
from glob import glob
import time
import os

class Conv1d:
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, lr=0.001):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.lr = lr
        # He / Kaiming initialization
        scale = np.sqrt(2.0 / (in_channels * kernel_size))
        self.kernel = np.random.randn(out_channels, in_channels, kernel_size) * scale
        self.bias = np.zeros(out_channels)

    def forward(self, x):
        batch, in_c, length = x.shape
        if in_c != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} channels, got {in_c}")

        out_len = (length + 2 * self.padding - self.kernel_size) // self.stride + 1
        out = np.zeros((batch, self.out_channels, out_len))
        x_pad = np.pad(x, ((0,0), (0,0), (self.padding, self.padding)), mode='constant')

        for b in range(batch):
            for o in range(self.out_channels):
                for i in range(out_len):
                    start = i * self.stride
                    window = x_pad[b, :, start:start+self.kernel_size]
                    out[b, o, i] = np.sum(window * self.kernel[o]) + self.bias[o]

        self.x_pad = x_pad
        self.out_len = out_len
        return out

    def backward(self, grad_out):
        batch, out_c, out_len = grad_out.shape
        grad_kernel = np.zeros_like(self.kernel)
        grad_bias = np.sum(grad_out, axis=(0,2))
        grad_x_pad = np.zeros_like(self.x_pad)

        for b in range(batch):
            for o in range(out_c):
                for i in range(out_len):
                    start = i * self.stride
                    grad_kernel[o] += grad_out[b, o, i] * self.x_pad[b, :, start:start+self.kernel_size]
                    grad_x_pad[b, :, start:start+self.kernel_size] += grad_out[b, o, i] * self.kernel[o]

        self.kernel -= self.lr * grad_kernel
        self.bias -= self.lr * grad_bias

        if self.padding > 0:
            grad_input = grad_x_pad[:, :, self.padding:-self.padding]
        else:
            grad_input = grad_x_pad
        return grad_input

    def set_lr(self, lr):
        self.lr = lr

class ConvTranspose1d:
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, output_padding=0, lr=0.001):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.output_padding = output_padding
        self.lr = lr
        scale = np.sqrt(2.0 / (in_channels * kernel_size))
        self.kernel = np.random.randn(in_channels, out_channels, kernel_size) * scale
        self.bias = np.zeros(out_channels)

    def forward(self, x):
        batch, in_c, in_len = x.shape
        out_len = (in_len - 1) * self.stride + self.kernel_size - 2 * self.padding + self.output_padding
        out = np.zeros((batch, self.out_channels, out_len))

        x_dilated = np.zeros((batch, in_c, (in_len-1)*self.stride + 1))
        for b in range(batch):
            for c in range(in_c):
                x_dilated[b, c, ::self.stride] = x[b, c]

        for b in range(batch):
            for o in range(self.out_channels):
                for i in range(in_len):
                    start = i * self.stride
                    for k in range(self.kernel_size):
                        pos = start + k - self.padding
                        if 0 <= pos < out_len:
                            out[b, o, pos] += np.sum(x_dilated[b, :, start] * self.kernel[:, o, k]) + self.bias[o]
        self.x = x
        self.out_len = out_len
        return out

    def backward(self, grad_out):
        batch, out_c, out_len = grad_out.shape
        grad_kernel = np.zeros_like(self.kernel)
        grad_bias = np.sum(grad_out, axis=(0,2))
        grad_x = np.zeros_like(self.x)

        in_len = self.x.shape[2]
        x_dilated = np.zeros((batch, self.in_channels, (in_len-1)*self.stride + 1))
        for b in range(batch):
            for c in range(self.in_channels):
                x_dilated[b, c, ::self.stride] = self.x[b, c]

        for b in range(batch):
            for o in range(out_c):
                for i in range(in_len):
                    start = i * self.stride
                    for k in range(self.kernel_size):
                        pos = start + k - self.padding
                        if 0 <= pos < out_len:
                            grad_kernel[:, o, k] += grad_out[b, o, pos] * x_dilated[b, :, start]
                            grad_x[b, :, i] += grad_out[b, o, pos] * self.kernel[:, o, k]

        self.kernel -= self.lr * grad_kernel
        self.bias -= self.lr * grad_bias
        return grad_x

    def set_lr(self, lr):
        self.lr = lr

class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, grad_out):
        return grad_out * (self.x > 0).astype(float)

class Encoder:
    def __init__(self, lr=0.001):
        self.conv1 = Conv1d(1, 32, kernel_size=7, stride=2, padding=1, lr=lr)
        self.relu1 = ReLU()
        self.conv2 = Conv1d(32, 64, kernel_size=7, stride=2, padding=1, lr=lr)
        self.relu2 = ReLU()
        self.conv3 = Conv1d(64, 128, kernel_size=7, stride=2, padding=1, lr=lr)
        self.relu3 = ReLU()

    def forward(self, x):
        out = self.conv1.forward(x)
        out = self.relu1.forward(out)
        out = self.conv2.forward(out)
        out = self.relu2.forward(out)
        out = self.conv3.forward(out)
        out = self.relu3.forward(out)
        return out

    def backward(self, grad_out):
        grad = self.relu3.backward(grad_out)
        grad = self.conv3.backward(grad)
        grad = self.relu2.backward(grad)
        grad = self.conv2.backward(grad)
        grad = self.relu1.backward(grad)
        grad = self.conv1.backward(grad)
        return grad

class VectorQuantization:
    def __init__(self, num_embeddings, embedding_dim, lr=0.01):
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.lr = lr
        # Initialize codebook uniformly
        self.codebook = np.random.randn(embedding_dim, num_embeddings)
        self.codebook /= np.linalg.norm(self.codebook, axis=0, keepdims=True)

    def forward(self, z_e):
        batch, dim, length = z_e.shape
        z_e_flat = z_e.transpose(0,2,1).reshape(-1, dim)

        # Compute distances
        z_e_sq = np.sum(z_e_flat**2, axis=1, keepdims=True)
        codebook_sq = np.sum(self.codebook**2, axis=0, keepdims=True)
        dist = z_e_sq + codebook_sq - 2 * z_e_flat @ self.codebook

        indices = np.argmin(dist, axis=1)
        z_q = self.codebook[:, indices].T
        z_q = z_q.reshape(batch, length, dim).transpose(0,2,1)

        self.z_e = z_e
        self.z_q = z_q
        self.indices = indices
        self.batch = batch
        self.length = length
        return z_q, indices

    def backward(self, grad_out):
        # Straight-through estimator
        grad_e = grad_out.copy()

        z_e_flat = self.z_e.transpose(0,2,1).reshape(-1, self.embedding_dim)
        z_q_flat = self.z_q.transpose(0,2,1).reshape(-1, self.embedding_dim)

        d_codebook = np.zeros_like(self.codebook)
        for i, idx in enumerate(self.indices):
            d_codebook[:, idx] += z_q_flat[i] - z_e_flat[i]

        self.codebook -= self.lr * d_codebook / (self.batch * self.length)
        return grad_e

class Decoder:
    def __init__(self, lr=0.001):
        self.conv1 = ConvTranspose1d(128, 64, kernel_size=7, stride=2, padding=1, output_padding=0, lr=lr)
        self.relu1 = ReLU()
        self.conv2 = ConvTranspose1d(64, 32, kernel_size=7, stride=2, padding=1, output_padding=1, lr=lr)
        self.relu2 = ReLU()
        self.conv3 = ConvTranspose1d(32, 1,  kernel_size=7, stride=2, padding=1, output_padding=1, lr=lr)
        self.relu3 = ReLU()

    def forward(self, z_q):
        out = self.conv1.forward(z_q)
        out = self.relu1.forward(out)
        out = self.conv2.forward(out)
        out = self.relu2.forward(out)
        out = self.conv3.forward(out)
        out = self.relu3.forward(out)
        return out

    def backward(self, grad_out):
        grad = self.relu3.backward(grad_out)
        grad = self.conv3.backward(grad)
        grad = self.relu2.backward(grad)
        grad = self.conv2.backward(grad)
        grad = self.relu1.backward(grad)
        grad = self.conv1.backward(grad)
        return grad

class NeuralCodec:
    def __init__(self, num_embeddings=512, embedding_dim=128, lr=0.001):
        self.encoder = Encoder(lr=lr)
        self.decoder = Decoder(lr=lr)
        self.vq = VectorQuantization(num_embeddings, embedding_dim, lr=lr)

    def forward(self, x):
        z_e = self.encoder.forward(x)
        z_q, indices = self.vq.forward(z_e)
        recon = self.decoder.forward(z_q)
        return recon, z_q, z_e, indices

    def compute_loss(self, recon, x, z_q, z_e):
        recon_loss = np.mean((x - recon) ** 2)
        vq_loss = np.mean((z_q - z_e) ** 2)
        return recon_loss + vq_loss

    def backward(self, x, recon, z_q, z_e):
        grad_recon = 2 * (recon - x) / np.prod(x.shape)
        grad_dec = self.decoder.backward(grad_recon)
        grad_vq = self.vq.backward(grad_dec)
        grad_enc = self.encoder.backward(grad_vq)
        return grad_enc

    def train_step(self, x):
        recon, z_q, z_e, _ = self.forward(x)
        loss = self.compute_loss(recon, x, z_q, z_e)
        self.backward(x, recon, z_q, z_e)
        return loss

    #save the parameter in case google colab is disconnected
    def save(self, save_dir, epoch=None):
        os.makedirs(save_dir, exist_ok=True)
        np.savez(os.path.join(save_dir, f"encoder_epoch{epoch}.npz" if epoch else "encoder_final.npz"),
                 conv1_kernel=self.encoder.conv1.kernel, conv1_bias=self.encoder.conv1.bias,
                 conv2_kernel=self.encoder.conv2.kernel, conv2_bias=self.encoder.conv2.bias,
                 conv3_kernel=self.encoder.conv3.kernel, conv3_bias=self.encoder.conv3.bias)
        np.savez(os.path.join(save_dir, f"decoder_epoch{epoch}.npz" if epoch else "decoder_final.npz"),
                 conv1_kernel=self.decoder.conv1.kernel, conv1_bias=self.decoder.conv1.bias,
                 conv2_kernel=self.decoder.conv2.kernel, conv2_bias=self.decoder.conv2.bias,
                 conv3_kernel=self.decoder.conv3.kernel, conv3_bias=self.decoder.conv3.bias)
        np.save(os.path.join(save_dir, f"vq_codebook_epoch{epoch}.npy" if epoch else "vq_codebook.npy"),
                self.vq.codebook)

    #Load the weight to the model again
    def load(self, save_dir, epoch=None):
        enc_file = os.path.join(save_dir, f"encoder_epoch{epoch}.npz" if epoch else "encoder_final.npz")
        dec_file = os.path.join(save_dir, f"decoder_epoch{epoch}.npz" if epoch else "decoder_final.npz")
        vq_file = os.path.join(save_dir, f"vq_codebook_epoch{epoch}.npy" if epoch else "vq_codebook.npy")

        if os.path.exists(enc_file):
            data = np.load(enc_file)
            self.encoder.conv1.kernel = data['conv1_kernel']
            self.encoder.conv1.bias = data['conv1_bias']
            self.encoder.conv2.kernel = data['conv2_kernel']
            self.encoder.conv2.bias = data['conv2_bias']
            self.encoder.conv3.kernel = data['conv3_kernel']
            self.encoder.conv3.bias = data['conv3_bias']

        if os.path.exists(dec_file):
            data = np.load(dec_file)
            self.decoder.conv1.kernel = data['conv1_kernel']
            self.decoder.conv1.bias = data['conv1_bias']
            self.decoder.conv2.kernel = data['conv2_kernel']
            self.decoder.conv2.bias = data['conv2_bias']
            self.decoder.conv3.kernel = data['conv3_kernel']
            self.decoder.conv3.bias = data['conv3_bias']

        if os.path.exists(vq_file):
            self.vq.codebook = np.load(vq_file)

#Load the data from audio
class AudioDataset:
    def __init__(self, data_dir, sample_rate=8000, segment_length=2048, max_files=None):
        self.sample_rate = sample_rate
        self.segment_length = segment_length
        # Recursively find all .wav files in subdirectories
        self.file_list = glob(os.path.join(data_dir, "**/*.wav"), recursive=True)

        if max_files is not None:
            self.file_list = self.file_list[:max_files]
        print(f"Found {len(self.file_list)} audio files for training.")

        self.audio_data = []
        for fpath in self.file_list:
            try:
                y, sr = librosa.load(fpath, sr=sample_rate, mono=True)
                y = y / (np.max(np.abs(y)) + 1e-8) # Normalize audio
                self.audio_data.append(y)
            except Exception as e:
                pass

    def __len__(self):
        return len(self.audio_data)


    def __getitem__(self, idx):
        y = self.audio_data[idx]
        #if the length of y is too short, pad it
        if len(y) < self.segment_length:
            y = np.pad(y, (0, self.segment_length - len(y)))
        #set random start point to avoid overfitting
        start = np.random.randint(0, len(y) - self.segment_length + 1)
        segment = y[start:start+self.segment_length]
        return segment.reshape(1, -1)

def create_batches(dataset, batch_size):
    indices = np.random.permutation(len(dataset))
    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start+batch_size]
        batch = [dataset[i] for i in batch_indices]
        batch_x = np.stack(batch, axis=0)
        yield batch_x

In [ ]:
#Training loop

def train_model(model, dataset, batch_size, epochs, save_dir, checkpoint_interval=5):
    os.makedirs(save_dir, exist_ok=True)
    loss_history = []
    num_batches = len(dataset) // batch_size + (1 if len(dataset) % batch_size != 0 else 0)

    print("\n--- INITIATING BALANCED TRAINING PROCESS ---")
    print(f"Total Samples: {len(dataset)} (Balanced: 200 Cats / 200 Dogs)")

    for epoch in range(1, epochs + 1):
        epoch_losses = []
        # Create random batches for each epoch
        batch_gen = create_batches(dataset, batch_size)
        start_time = time.time()

        for batch_idx, batch_x in enumerate(batch_gen):
            loss = model.train_step(batch_x)
            epoch_losses.append(loss)

            if batch_idx % 10 == 0:
                print(f"Epoch {epoch}/{epochs} | Batch {batch_idx}/{num_batches} | Current Loss: {loss:.6f}")

        avg_loss = np.mean(epoch_losses)
        loss_history.append(avg_loss)
        elapsed = time.time() - start_time
        print(f">>> EPOCH {epoch} COMPLETED | Average Loss: {avg_loss:.6f} | Time: {elapsed:.2f}s")

        if epoch % checkpoint_interval == 0:
            model.save(save_dir, epoch=epoch)

    model.save(save_dir, epoch="final")
    print("TRAINING COMPLETED")
    return loss_history


import os
from glob import glob

print("Scanning for audio folders...")
cat_files_temp = glob("/content/dataset/**/cat*.wav", recursive=True)
dog_files_temp = glob("/content/dataset/**/dog*.wav", recursive=True)

if not cat_files_temp or not dog_files_temp:
    raise ValueError("CRITICAL ERROR: Could not find audio files")

CAT_DIR = os.path.dirname(cat_files_temp[0])
DOG_DIR = os.path.dirname(dog_files_temp[0])
SAVE_DIR = "/content/drive/MyDrive/Numpy_Codec_Weights_Balanced"

print(f"-> CAT_DIR: {CAT_DIR}")
print(f"-> DOG_DIR: {DOG_DIR}")

cat_data = AudioDataset(data_dir=CAT_DIR, sample_rate=8000, segment_length=2048, max_files=400)
dog_data = AudioDataset(data_dir=DOG_DIR, sample_rate=8000, segment_length=2048, max_files=400)

# Merge datasets
dataset = dog_data
dataset.audio_data.extend(cat_data.audio_data)
print(f"Balanced Dataset Ready: {len(dataset.audio_data)} samples in total.")

#Initialize model
model = NeuralCodec(
    num_embeddings=128,
    embedding_dim=128,
    lr=0.001)

#Load pretrained weight from previous time before training
WEIGHTS_PATH = "/content/drive/MyDrive/Numpy_Codec_Weights_Balanced"
try:
    print("Attempting to load pre-trained weights...")
    model.load(WEIGHTS_PATH, epoch="final")
    print(" SUCCESS: Pre-trained weights loaded! Continuing training from existing knowledge.")
except Exception as e:
    print(f" WARNING: Could not load weights ({e}). The model will start learning from scratch.")


#Start training
loss_history = train_model(
    model=model,
    dataset=dataset,
    batch_size=4,
    epochs=10,
    save_dir=WEIGHTS_PATH,
    checkpoint_interval=5
)